# 04 — Modelling handoff export
\

In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

# Handle notebooks running from the notebooks folder
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS = PROJECT_ROOT / 'reports' / 'modelling_handoff'
REPORTS.mkdir(parents=True, exist_ok=True)

source = PROCESSED / 'listings_features.parquet'
if not source.exists():
    raise FileNotFoundError(f'Feature-engineering output not found: {source}.')

features = pd.read_parquet(source).copy()
required = {'id','location','country','price_numeric'}
missing = required - set(features.columns)
if missing:
    raise ValueError(f'Required fields missing: {sorted(missing)}')

# Do not silently discard or invent data when listing keys are duplicated.
key = ['location', 'id']
null_keys = features[key].isna().any(axis=1)
duplicated = features.duplicated(key, keep=False)
if null_keys.any() or duplicated.any():
    exception_path = REPORTS / 'listing_key_exceptions.csv'
    features.loc[null_keys | duplicated, ['country','location','id']].to_csv(exception_path, index=False)
    raise ValueError(f'Listing keys need investigation: {int(null_keys.sum())} missing, '
                     f'{int(duplicated.sum())} duplicate rows. See {exception_path}')

expected_currency = {
    'Austria':'EUR','Belgium':'EUR','Czech Republic':'CZK',
    'Denmark':'DKK','France':'EUR','Germany':'EUR','Greece':'EUR',
    'Ireland':'EUR','Italy':'EUR','Latvia':'EUR','Netherlands':'EUR',
    'Norway':'NOK','Portugal':'EUR','Spain':'EUR','Switzerland':'CHF',
    'United Kingdom':'GBP'
}
features['expected_currency'] = features['country'].map(expected_currency).astype('string')
features['price_numeric'] = pd.to_numeric(features['price_numeric'], errors='coerce')
features['price_missing'] = features['price_numeric'].isna()
features['price_nonpositive'] = features['price_numeric'].notna() & (features['price_numeric'] <= 0)
features['price_target_valid'] = (~features['price_missing']) & (~features['price_nonpositive']) & np.isfinite(features['price_numeric'].to_numpy(dtype='float64', na_value=np.nan))
features['price_scale_unresolved'] = features['location'].isin(['Geneva','Vaud'])

# Internal numeric agreement and expected local currency are not independent currency verification.
features['currency_verification_status'] = 'UNVERIFIED_EXPECTED_LOCAL'
features.loc[features['expected_currency'].isna(), 'currency_verification_status'] = 'UNKNOWN'
features.loc[features['price_scale_unresolved'], 'currency_verification_status'] = 'SCALING_UNRESOLVED'

if 'verification_status' in features.columns:
    # Retain previous audit status, but do not upgrade to verified automatically.
    features['previous_currency_audit_status'] = features['verification_status'].astype('string')

# Full preserved feature dataset, including rows without a valid price.
full_path = PROCESSED / 'listings_features_final.parquet'
features.to_parquet(full_path, index=False, compression='zstd')

# Candidate for *within-market* modelling, never automatically for pooled currency predictions.
# Keep an explicit whitelist; do not accidentally include prices, benchmarks, ID or target proxies.
predictor_whitelist = [
    'room_type','property_type','accommodates_clean','bedrooms_clean','beds_clean',
    'bathrooms_numeric','amenities_count','has_wifi','has_air_conditioning',
    'has_kitchen','has_parking','has_pool','has_workspace',
    'minimum_nights_clean','is_superhost','latitude','longitude',
    'competitors_500m','competitors_1km','competitors_2km',
    'comparable_competitors_1km','neighbourhood_listing_count',
    'neighbourhood_host_count','location_listing_count',
]
predictors = [x for x in predictor_whitelist if x in features.columns]
metadata = ['country','location','id','expected_currency','currency_verification_status']
eligible = features['price_target_valid'] & ~features['price_scale_unresolved'] & features['expected_currency'].notna() & features["country"].ne("Switzerland").fillna(False)
model_candidate = features.loc[eligible, metadata + predictors].copy()
model_candidate.insert(len(metadata), 'target_price_local', features.loc[eligible, 'price_numeric'].astype('float64').to_numpy())
candidate_path = PROCESSED / 'model_candidate.parquet'
model_candidate.to_parquet(candidate_path, index=False, compression='zstd')

# Analysis-only measures: target-derived features stay out of the modelling predictor whitelist.
analysis_cols = [
    'country','location','id','expected_currency','currency_verification_status',
    'price_numeric','price_target_valid','price_scale_unresolved',
    'room_type','latitude','longitude','reviews_last_30d','reviews_last_90d',
    'reviews_last_365d','days_since_last_review','availability_rate_365',
    'competitors_500m','competitors_1km','competitors_2km',
    'comparable_competitors_1km','neighbourhood_listing_count',
    'neighbourhood_host_count','market_median_price',
    'neighbourhood_median_price','price_vs_neighbourhood_pct'
]
analysis_cols = [x for x in analysis_cols if x in features.columns]
analysis_path = PROCESSED / 'market_analysis.parquet'
features[analysis_cols].to_parquet(analysis_path, index=False, compression='zstd')

# Audit coverage by market.
audit = (features.assign(model_candidate_eligible=eligible)
    .groupby(['country','location','expected_currency','currency_verification_status'], dropna=False)
    .agg(total_listings=('id','size'), missing_price=('price_missing','sum'),
         nonpositive_price=('price_nonpositive','sum'),
         valid_target=('price_target_valid','sum'),
         model_candidate_rows=('model_candidate_eligible','sum'))
    .reset_index())
audit['missing_price_pct'] = (100 * audit['missing_price'] / audit['total_listings']).round(2)
audit.to_csv(REPORTS / 'modelling_handoff_report.csv', index=False)

model_spec = {
    'dataset':'model_candidate.parquet',
    'status':'PROVISIONAL — NOT APPROVED FOR POOLED CROSS-MARKET PRICE MODELLING',
    'target':'target_price_local',
    'predictors':predictors,
    'metadata_not_predictors':metadata,
    'excluded_from_candidate':['Geneva','Vaud'],
    'currency_policy':'Expected local currencies only; no FX conversion or independent source verification performed.',
    'training_note':'For initial modelling, select ONE market with source price units verified. Do not pool locations in different currencies.',
    'leakage_policy':'Price-derived benchmarks, price_numeric, price_quote_price_per_night and price_vs_neighbourhood_pct are excluded from predictors. Historical review timing not verified; review activity omitted from baseline predictors.',
    'additional_checks':'Resolve cross-location overlaps; verify snapshot time alignment, missingness and sampling bias before final training.',
}
with open(REPORTS / 'modelling_feature_list.json', 'w', encoding='utf-8') as f:
    json.dump(model_spec, f, indent=2, ensure_ascii=False)

print(f'Source: {len(features):,} listing rows')
print(f'Model candidate: {len(model_candidate):,} rows, {len(predictors)} predictors')
for path in [full_path,candidate_path,analysis_path,REPORTS/'modelling_feature_list.json',REPORTS/'modelling_handoff_report.csv']:
    print('Saved:',path)

Source: 756,312 listing rows
Model candidate: 605,976 rows, 24 predictors
Saved: C:\Users\erins\OneDrive\Documents\Airbnb_VSCode_Project\Airbnb_EDACode_Project_\reordered notebooks\data\processed\listings_features_final.parquet
Saved: C:\Users\erins\OneDrive\Documents\Airbnb_VSCode_Project\Airbnb_EDACode_Project_\reordered notebooks\data\processed\model_candidate.parquet
Saved: C:\Users\erins\OneDrive\Documents\Airbnb_VSCode_Project\Airbnb_EDACode_Project_\reordered notebooks\data\processed\market_analysis.parquet
Saved: C:\Users\erins\OneDrive\Documents\Airbnb_VSCode_Project\Airbnb_EDACode_Project_\reordered notebooks\reports\modelling_handoff\modelling_feature_list.json
Saved: C:\Users\erins\OneDrive\Documents\Airbnb_VSCode_Project\Airbnb_EDACode_Project_\reordered notebooks\reports\modelling_handoff\modelling_handoff_report.csv


In [5]:
# Ensure Switzerland has been excluded
assert not model_candidate["country"].eq("Switzerland").any()

# Ensure all target prices are valid
assert model_candidate["target_price_local"].notna().all()
assert (model_candidate["target_price_local"] > 0).all()

# Check country distribution
display(
    model_candidate.groupby(
        ["country", "expected_currency"]
    ).size().reset_index(name="listings")
)

print("Validation passed: Switzerland excluded.")

,country,expected_currency,listings
0,Austria,EUR,10492
1,Belgium,EUR,7658
2,Czech Republic,CZK,10583
3,Denmark,DKK,13860
4,France,EUR,62242
5,Germany,EUR,12881
6,Greece,EUR,78783
7,Ireland,EUR,5578
8,Italy,EUR,192497
9,Latvia,EUR,3149


Validation passed: Switzerland excluded.
